## 1. Imports

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

## 2. Paths

In [2]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

ANALYTICS_PATH = os.path.join(
    PROCESSED_PATH,
    "company_analytics"
)

PROFILE_PATH = os.path.join(
    PROCESSED_PATH,
    "company_profiles"
)

TIMELINE_PATH = os.path.join(
    PROCESSED_PATH,
    "company_timeline"
)

GRAPH_PATH = os.path.join(
    PROCESSED_PATH,
    "knowledge_graph"
)

os.makedirs(
    GRAPH_PATH,
    exist_ok=True
)

## 3. Load Datasets

In [3]:
company_profiles = pd.read_parquet(

    os.path.join(

        PROFILE_PATH,

        "company_profiles.parquet"

    )

)

timeline_df = pd.read_parquet(

    os.path.join(

        TIMELINE_PATH,

        "company_event_timeline.parquet"

    )

)

analytics = pd.read_parquet(

    os.path.join(

        ANALYTICS_PATH,

        "company_analytics.parquet"

    )

)

company_similarity = pd.read_parquet(

    os.path.join(

        ANALYTICS_PATH,

        "company_similarity.parquet"

    )

)

company_clusters = pd.read_parquet(

    os.path.join(

        ANALYTICS_PATH,

        "company_clusters.parquet"

    )

)

print("Company Profiles")
print(company_profiles.shape)

print()

print("Timeline")
print(timeline_df.shape)

print()

print("Analytics")
print(analytics.shape)

print()

print("Similarity")
print(company_similarity.shape)

print()

print("Clusters")
print(company_clusters.shape)

Company Profiles
(6619, 60)

Timeline
(3215296, 26)

Analytics
(6619, 70)

Similarity
(66190, 3)

Clusters
(12, 6)


## 4. Create Company Nodes

In [4]:
# Company Nodes
company_nodes = analytics[

    [

        "ticker",

        "overall_rank",

        "composite_score",

        "normalized_risk_score",

        "market_stability_score",

        "activity_score",

        "company_fingerprint"

    ]

].copy()

company_nodes = company_nodes.rename(

    columns={

        "ticker":"node_name"

    }

)

company_nodes["node_type"] = "Company"

company_nodes["node_id"] = (

    "COMP_"

    +

    company_nodes["node_name"].astype(str)

)

company_nodes = company_nodes[

    [

        "node_id",

        "node_name",

        "node_type",

        "overall_rank",

        "composite_score",

        "normalized_risk_score",

        "market_stability_score",

        "activity_score",

        "company_fingerprint"

    ]

]

print(company_nodes.shape)

company_nodes.head()

(6619, 9)


,node_id,node_name,node_type,overall_rank,composite_score,normalized_risk_score,market_stability_score,activity_score,company_fingerprint
0,COMP_GILD,GILD,Company,1,89.41,10.29,96.41,83.09,"Stable, High Activity, Bullish, Diversified"
1,COMP_HD,HD,Company,2,84.88,29.29,96.08,74.00,"High Activity, Bullish, Diversified"
2,COMP_MDT,MDT,Company,3,83.69,0.00,97.71,57.43,"Stable, Bullish, Diversified"
3,COMP_SLV,SLV,Company,4,81.65,41.21,92.37,72.31,"High Activity, Bullish, Diversified"
4,COMP_LOW,LOW,Company,5,80.88,23.55,96.09,59.47,"Bullish, Diversified"


## 5. Publisher Nodes

In [5]:
# Publisher Nodes
publisher_nodes = (

    timeline_df

    .groupby("publisher")

    .agg(

        total_articles=("news_id", "count"),

        companies=("ticker", "nunique"),

        avg_confidence=("final_confidence", "mean")

    )

    .reset_index()

)

publisher_nodes = publisher_nodes.rename(

    columns={

        "publisher": "node_name"

    }

)

publisher_nodes["node_type"] = "Publisher"

publisher_nodes["node_id"] = (

    "PUB_"

    +

    publisher_nodes["node_name"]

    .astype(str)

    .str.replace(" ", "_", regex=False)

)

publisher_nodes = publisher_nodes[

    [

        "node_id",

        "node_name",

        "node_type",

        "total_articles",

        "companies",

        "avg_confidence"

    ]

]

print(publisher_nodes.shape)

publisher_nodes.head()

(1047, 6)


,node_id,node_name,node_type,total_articles,companies,avg_confidence
0,PUB_47ertrends,47ertrends,Publisher,18,9,0.772850
1,PUB_AARP,AARP,Publisher,7,7,0.744371
2,PUB_ABNNewswire,ABNNewswire,Publisher,13,10,0.847838
3,PUB_Aakin,Aakin,Publisher,5,5,0.745940
4,PUB_Aaron_Jackson.Ed,Aaron Jackson.Ed,Publisher,16,16,0.744431


## 6. Event Nodes

In [6]:
# Event Nodes
event_nodes = (

    timeline_df

    .groupby("final_event")

    .agg(

        headlines=("news_id", "count"),

        avg_confidence=("final_confidence", "mean")

    )

    .reset_index()

)

event_nodes = event_nodes.rename(

    columns={

        "final_event":"node_name"

    }

)

event_nodes["node_type"] = "Event"

event_nodes["node_id"] = (

    "EVENT_"

    +

    event_nodes["node_name"]

    .str.replace(" ", "_", regex=False)

)

event_nodes = event_nodes[

    [

        "node_id",

        "node_name",

        "node_type",

        "headlines",

        "avg_confidence"

    ]

]

print(event_nodes.shape)

event_nodes.head()

(30, 5)


,node_id,node_name,node_type,headlines,avg_confidence
0,EVENT_Analyst_Rating,Analyst Rating,Event,277248,0.899277
1,EVENT_Bankruptcy,Bankruptcy,Event,3043,0.904568
2,EVENT_Commodity,Commodity,Event,176547,0.840405
3,EVENT_Credit_Rating,Credit Rating,Event,36358,0.956756
4,EVENT_Cryptocurrency,Cryptocurrency,Event,13739,0.773372


## 7. Market Signal Nodes

In [7]:
# Market Signal Nodes
signal_nodes = pd.DataFrame({

    "node_name":[

        "Bullish",

        "Bearish",

        "Neutral",

        "Mixed"

    ]

})

signal_nodes["node_type"] = "MarketSignal"

signal_nodes["node_id"] = (

    "SIGNAL_"

    +

    signal_nodes["node_name"]

)

signal_nodes

,node_name,node_type,node_id
0,Bullish,MarketSignal,SIGNAL_Bullish
1,Bearish,MarketSignal,SIGNAL_Bearish
2,Neutral,MarketSignal,SIGNAL_Neutral
3,Mixed,MarketSignal,SIGNAL_Mixed


## 8. Cluster Nodes

In [8]:
# Cluster Nodes
cluster_nodes = company_clusters.copy()

cluster_nodes["node_name"] = (

    "Cluster_"

    +

    cluster_nodes["cluster"].astype(str)

)

cluster_nodes["node_type"] = "Cluster"

cluster_nodes["node_id"] = (

    "CLUSTER_"

    +

    cluster_nodes["cluster"].astype(str)

)

cluster_nodes = cluster_nodes[

    [

        "node_id",

        "node_name",

        "node_type",

        "companies",

        "avg_activity",

        "avg_risk",

        "avg_score",

        "category"

    ]

]

print(cluster_nodes.shape)

cluster_nodes.head()

(12, 8)


,node_id,node_name,node_type,companies,avg_activity,avg_risk,avg_score,category
0,CLUSTER_0,Cluster_0,Cluster,1020,21.77,34.20,65.63,Balanced
1,CLUSTER_1,Cluster_1,Cluster,994,4.78,38.19,52.15,Balanced
2,CLUSTER_2,Cluster_2,Cluster,944,16.01,37.62,62.20,Balanced
3,CLUSTER_3,Cluster_3,Cluster,148,43.74,35.22,74.38,Balanced
4,CLUSTER_4,Cluster_4,Cluster,430,31.79,33.72,70.17,Balanced


## 9. Fingerprint Nodes

In [9]:
# Fingerprint Nodes
fingerprint_nodes = (

    analytics[

        [

            "company_fingerprint"

        ]

    ]

    .drop_duplicates()

    .reset_index(drop=True)

)

fingerprint_nodes = fingerprint_nodes.rename(

    columns={

        "company_fingerprint":"node_name"

    }

)

fingerprint_nodes["node_type"] = "Fingerprint"

fingerprint_nodes["node_id"] = (

    "FP_"

    +

    fingerprint_nodes.index.astype(str)

)

print(fingerprint_nodes.shape)

fingerprint_nodes.head()

(8, 3)


,node_name,node_type,node_id
0,"Stable, High Activity, Bullish, Diversified",Fingerprint,FP_0
1,"High Activity, Bullish, Diversified",Fingerprint,FP_1
2,"Stable, Bullish, Diversified",Fingerprint,FP_2
3,"Bullish, Diversified",Fingerprint,FP_3
4,"High Risk, Bullish, Diversified",Fingerprint,FP_4


## 10. Merge All Nodes

In [10]:
# Merge All Nodes
nodes = pd.concat(

    [

        company_nodes,

        publisher_nodes,

        event_nodes,

        signal_nodes,

        cluster_nodes,

        fingerprint_nodes

    ],

    ignore_index=True,

    sort=False

)

print("Knowledge Graph Nodes")

print(nodes.shape)

display(

    nodes.head()

)

Knowledge Graph Nodes
(7720, 17)


,node_id,node_name,node_type,overall_rank,composite_score,normalized_risk_score,market_stability_score,activity_score,company_fingerprint,total_articles,companies,avg_confidence,headlines,avg_activity,avg_risk,avg_score,category
0,COMP_GILD,GILD,Company,1.0,89.41,10.29,96.41,83.09,"Stable, High Activity, Bullish, Diversified",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,COMP_HD,HD,Company,2.0,84.88,29.29,96.08,74.00,"High Activity, Bullish, Diversified",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,COMP_MDT,MDT,Company,3.0,83.69,0.00,97.71,57.43,"Stable, Bullish, Diversified",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,COMP_SLV,SLV,Company,4.0,81.65,41.21,92.37,72.31,"High Activity, Bullish, Diversified",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,COMP_LOW,LOW,Company,5.0,80.88,23.55,96.09,59.47,"Bullish, Diversified",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
nodes["display_label"] = nodes["node_name"]

### Clean Node Attributes

In [12]:
# Clean Node Attributes
numeric_columns = nodes.select_dtypes(

    include=["number"]

).columns

nodes[numeric_columns] = nodes[numeric_columns].fillna(0)

text_columns = nodes.select_dtypes(

    include=["object"]

).columns

nodes[text_columns] = nodes[text_columns].fillna("")

print(nodes.isna().sum().sum())

0


In [13]:
print("Knowledge Graph Nodes")

print(nodes.shape)

display(nodes.head())

Knowledge Graph Nodes
(7720, 18)


,node_id,node_name,node_type,overall_rank,composite_score,normalized_risk_score,market_stability_score,activity_score,company_fingerprint,total_articles,companies,avg_confidence,headlines,avg_activity,avg_risk,avg_score,category,display_label
0,COMP_GILD,GILD,Company,1.0,89.41,10.29,96.41,83.09,"Stable, High Activity, Bullish, Diversified",0.0,0.0,0.0,0.0,0.0,0.0,0.0,,GILD
1,COMP_HD,HD,Company,2.0,84.88,29.29,96.08,74.00,"High Activity, Bullish, Diversified",0.0,0.0,0.0,0.0,0.0,0.0,0.0,,HD
2,COMP_MDT,MDT,Company,3.0,83.69,0.00,97.71,57.43,"Stable, Bullish, Diversified",0.0,0.0,0.0,0.0,0.0,0.0,0.0,,MDT
3,COMP_SLV,SLV,Company,4.0,81.65,41.21,92.37,72.31,"High Activity, Bullish, Diversified",0.0,0.0,0.0,0.0,0.0,0.0,0.0,,SLV
4,COMP_LOW,LOW,Company,5.0,80.88,23.55,96.09,59.47,"Bullish, Diversified",0.0,0.0,0.0,0.0,0.0,0.0,0.0,,LOW


## 11. Company -> Publisher Edges

In [19]:
graph_df = timeline_df.copy()

graph_df["ticker"] = graph_df["ticker"].astype(str)
graph_df["publisher"] = graph_df["publisher"].astype(str)

company_publisher_edges = (

    graph_df

    .groupby(

        [

            "ticker",

            "publisher"

        ],

        observed=True

    )

    .agg(

        weight=("news_id", "count"),

        confidence=("final_confidence", "mean")

    )

    .reset_index()

)

company_publisher_edges["source"] = (

    "COMP_"

    +

    company_publisher_edges["ticker"]

)

company_publisher_edges["target"] = (

    "PUB_"

    +

    company_publisher_edges["publisher"]

        .str.replace(" ", "_", regex=False)

)

company_publisher_edges["relation"] = "PUBLISHED_BY"

company_publisher_edges = company_publisher_edges[

    [

        "source",

        "target",

        "relation",

        "weight",

        "confidence"

    ]

]

print("Company -> Publisher Edges")
print(company_publisher_edges.shape)

display(company_publisher_edges.head())

Company -> Publisher Edges
(201847, 5)


,source,target,relation,weight,confidence
0,COMP_A,PUB_Above_Average_Odds,PUBLISHED_BY,2,0.867500
1,COMP_A,PUB_Accesswire,PUBLISHED_BY,1,0.748400
2,COMP_A,PUB_Allie_Wickman,PUBLISHED_BY,27,0.830659
3,COMP_A,PUB_Andrew_Efimoff,PUBLISHED_BY,2,0.866500
4,COMP_A,PUB_Angry_Bear,PUBLISHED_BY,2,0.738000


## 12. Company -> Event Edges

In [20]:
graph_df["final_event"] = graph_df["final_event"].astype(str)

company_event_edges = (

    graph_df

    .groupby(

        [

            "ticker",

            "final_event"

        ],

        observed=True

    )

    .agg(

        weight=("news_id", "count"),

        confidence=("final_confidence", "mean")

    )

    .reset_index()

)

company_event_edges["source"] = (

    "COMP_"

    +

    company_event_edges["ticker"]

)

company_event_edges["target"] = (

    "EVENT_"

    +

    company_event_edges["final_event"]

        .str.replace(" ", "_", regex=False)

)

company_event_edges["relation"] = "HAS_EVENT"

company_event_edges = company_event_edges[

    [

        "source",

        "target",

        "relation",

        "weight",

        "confidence"

    ]

]

print("Company -> Event Edges")
print(company_event_edges.shape)

display(company_event_edges.head())

Company -> Event Edges
(116520, 5)


,source,target,relation,weight,confidence
0,COMP_A,EVENT_Analyst_Rating,HAS_EVENT,207,0.904322
1,COMP_A,EVENT_Bankruptcy,HAS_EVENT,1,0.500000
2,COMP_A,EVENT_Commodity,HAS_EVENT,52,0.867298
3,COMP_A,EVENT_Credit_Rating,HAS_EVENT,26,0.908942
4,COMP_A,EVENT_Cryptocurrency,HAS_EVENT,3,0.734533


## 13. Company -> Market Signal Edges

In [21]:
graph_df["market_signal"] = graph_df["market_signal"].astype(str)

company_signal_edges = (

    graph_df

    .groupby(

        [

            "ticker",

            "market_signal"

        ],

        observed=True

    )

    .agg(

        weight=("news_id", "count"),

        confidence=("final_confidence", "mean")

    )

    .reset_index()

)

company_signal_edges["source"] = (

    "COMP_"

    +

    company_signal_edges["ticker"]

)

company_signal_edges["target"] = (

    "SIGNAL_"

    +

    company_signal_edges["market_signal"]

)

company_signal_edges["relation"] = "HAS_SIGNAL"

company_signal_edges = company_signal_edges[

    [

        "source",

        "target",

        "relation",

        "weight",

        "confidence"

    ]

]

print("Company -> Market Signal Edges")
print(company_signal_edges.shape)

display(company_signal_edges.head())

Company -> Market Signal Edges
(21314, 5)


,source,target,relation,weight,confidence
0,COMP_A,SIGNAL_Bearish,HAS_SIGNAL,46,0.859543
1,COMP_A,SIGNAL_Bullish,HAS_SIGNAL,309,0.947977
2,COMP_A,SIGNAL_Mixed,HAS_SIGNAL,10,0.903900
3,COMP_A,SIGNAL_Neutral,HAS_SIGNAL,1963,0.830047
4,COMP_AA,SIGNAL_Bearish,HAS_SIGNAL,83,0.938193


## 14. Company -> Cluster Edges

In [22]:
company_cluster_edges = analytics[

    [

        "ticker",

        "cluster"

    ]

].copy()

company_cluster_edges["ticker"] = company_cluster_edges["ticker"].astype(str)

company_cluster_edges["source"] = (

    "COMP_"

    +

    company_cluster_edges["ticker"]

)

company_cluster_edges["target"] = (

    "CLUSTER_"

    +

    company_cluster_edges["cluster"].astype(str)

)

company_cluster_edges["relation"] = "BELONGS_TO_CLUSTER"

company_cluster_edges["weight"] = 1

company_cluster_edges["confidence"] = 1.0

company_cluster_edges = company_cluster_edges[

    [

        "source",

        "target",

        "relation",

        "weight",

        "confidence"

    ]

]

print("Company -> Cluster Edges")
print(company_cluster_edges.shape)

display(company_cluster_edges.head())

Company -> Cluster Edges
(6619, 5)


,source,target,relation,weight,confidence
0,COMP_GILD,CLUSTER_3,BELONGS_TO_CLUSTER,1,1.0
1,COMP_HD,CLUSTER_3,BELONGS_TO_CLUSTER,1,1.0
2,COMP_MDT,CLUSTER_3,BELONGS_TO_CLUSTER,1,1.0
3,COMP_SLV,CLUSTER_11,BELONGS_TO_CLUSTER,1,1.0
4,COMP_LOW,CLUSTER_3,BELONGS_TO_CLUSTER,1,1.0


## 15. Company -> Fingerprint Edges

In [23]:
fingerprint_lookup = dict(

    zip(

        fingerprint_nodes["node_name"],

        fingerprint_nodes["node_id"]

    )

)

company_fingerprint_edges = analytics[

    [

        "ticker",

        "company_fingerprint"

    ]

].copy()

company_fingerprint_edges["ticker"] = company_fingerprint_edges["ticker"].astype(str)

company_fingerprint_edges["source"] = (

    "COMP_"

    +

    company_fingerprint_edges["ticker"]

)

company_fingerprint_edges["target"] = (

    company_fingerprint_edges["company_fingerprint"]

        .map(fingerprint_lookup)

)

company_fingerprint_edges["relation"] = "HAS_FINGERPRINT"

company_fingerprint_edges["weight"] = 1

company_fingerprint_edges["confidence"] = 1.0

company_fingerprint_edges = company_fingerprint_edges[

    [

        "source",

        "target",

        "relation",

        "weight",

        "confidence"

    ]

]

print("Company -> Fingerprint Edges")
print(company_fingerprint_edges.shape)

display(company_fingerprint_edges.head())

Company -> Fingerprint Edges
(6619, 5)


,source,target,relation,weight,confidence
0,COMP_GILD,FP_0,HAS_FINGERPRINT,1,1.0
1,COMP_HD,FP_1,HAS_FINGERPRINT,1,1.0
2,COMP_MDT,FP_2,HAS_FINGERPRINT,1,1.0
3,COMP_SLV,FP_1,HAS_FINGERPRINT,1,1.0
4,COMP_LOW,FP_3,HAS_FINGERPRINT,1,1.0


## 16. Company -> Similar Company

In [24]:
company_similarity_edges = company_similarity.copy()

company_similarity_edges["source"] = (

    "COMP_"

    +

    company_similarity_edges["ticker"].astype(str)

)

company_similarity_edges["target"] = (

    "COMP_"

    +

    company_similarity_edges["similar_company"].astype(str)

)

company_similarity_edges["relation"] = "SIMILAR_TO"

company_similarity_edges["weight"] = (

    company_similarity_edges["similarity_score"]

)

company_similarity_edges["confidence"] = (

    company_similarity_edges["similarity_score"]

)

company_similarity_edges = company_similarity_edges[

    [

        "source",

        "target",

        "relation",

        "weight",

        "confidence"

    ]

]

print("Company -> Similar Company")
print(company_similarity_edges.shape)

display(company_similarity_edges.head())

Company -> Similar Company
(66190, 5)


,source,target,relation,weight,confidence
0,COMP_GILD,COMP_LLY,SIMILAR_TO,0.9591,0.9591
1,COMP_GILD,COMP_BMY,SIMILAR_TO,0.9445,0.9445
2,COMP_GILD,COMP_REGN,SIMILAR_TO,0.9389,0.9389
3,COMP_GILD,COMP_MRK,SIMILAR_TO,0.9385,0.9385
4,COMP_GILD,COMP_MDT,SIMILAR_TO,0.9324,0.9324


## 17. Publisher -> Event

In [25]:
publisher_event_edges = (

    graph_df

    .groupby(

        [

            "publisher",

            "final_event"

        ],

        observed=True

    )

    .agg(

        weight=("news_id","count"),

        confidence=("final_confidence","mean")

    )

    .reset_index()

)

publisher_event_edges["source"] = (

    "PUB_"

    +

    publisher_event_edges["publisher"]

        .str.replace(" ", "_", regex=False)

)

publisher_event_edges["target"] = (

    "EVENT_"

    +

    publisher_event_edges["final_event"]

        .str.replace(" ", "_", regex=False)

)

publisher_event_edges["relation"] = "REPORTS"

publisher_event_edges = publisher_event_edges[

    [

        "source",

        "target",

        "relation",

        "weight",

        "confidence"

    ]

]

print("Publisher -> Event")
print(publisher_event_edges.shape)

display(publisher_event_edges.head())

Publisher -> Event
(7479, 5)


,source,target,relation,weight,confidence
0,PUB_47ertrends,EVENT_Market_Movement,REPORTS,18,0.772850
1,PUB_AARP,EVENT_Product_Launch,REPORTS,7,0.744371
2,PUB_ABNNewswire,EVENT_Analyst_Rating,REPORTS,2,1.000000
3,PUB_ABNNewswire,EVENT_Commodity,REPORTS,1,1.000000
4,PUB_ABNNewswire,EVENT_Funding,REPORTS,2,0.500000


## 18. Event -> Event

In [26]:
event_company = (

    graph_df

    [

        [

            "ticker",

            "final_event"

        ]

    ]

    .drop_duplicates()

)

merged = event_company.merge(

    event_company,

    on="ticker"

)

merged = merged[

    merged["final_event_x"]

    <

    merged["final_event_y"]

]

event_event_edges = (

    merged

    .groupby(

        [

            "final_event_x",

            "final_event_y"

        ],

        observed=True

    )

    .size()

    .reset_index(name="weight")

)

event_event_edges["source"] = (

    "EVENT_"

    +

    event_event_edges["final_event_x"]

        .str.replace(" ", "_", regex=False)

)

event_event_edges["target"] = (

    "EVENT_"

    +

    event_event_edges["final_event_y"]

        .str.replace(" ", "_", regex=False)

)

event_event_edges["relation"] = "CO_OCCURS"

event_event_edges["confidence"] = 1.0

event_event_edges = event_event_edges[

    [

        "source",

        "target",

        "relation",

        "weight",

        "confidence"

    ]

]

print("Event -> Event")
print(event_event_edges.shape)

display(event_event_edges.head())

Event -> Event
(435, 5)


,source,target,relation,weight,confidence
0,EVENT_Analyst_Rating,EVENT_Bankruptcy,CO_OCCURS,1093,1.0
1,EVENT_Analyst_Rating,EVENT_Commodity,CO_OCCURS,4303,1.0
2,EVENT_Analyst_Rating,EVENT_Credit_Rating,CO_OCCURS,3580,1.0
3,EVENT_Analyst_Rating,EVENT_Cryptocurrency,CO_OCCURS,2209,1.0
4,EVENT_Analyst_Rating,EVENT_Cybersecurity,CO_OCCURS,1731,1.0


## 19. Company Neighborhood

In [27]:
company_neighbors = (

    company_similarity

    .groupby("ticker")["similar_company"]

    .apply(list)

    .reset_index()

)

company_neighbors.columns = [

    "ticker",

    "neighbors"

]

company_neighbors["neighbor_count"] = (

    company_neighbors["neighbors"]

    .apply(len)

)

print("Company Neighborhood")

print(company_neighbors.shape)

display(company_neighbors.head())

Company Neighborhood
(6619, 3)


,ticker,neighbors,neighbor_count
0,A,"[CAG, INTU, MCHP, EL, TMO, IT, DGX, ADI, TXT, ...",10
1,AA,"[CAT, GPS, TRIP, M, COH, TSN, DAL, MRVL, NFLX,...",10
2,AAC,"[VMEM, ENT, RRTS, TST, ATEC, EIGI, MIND, WG, M...",10
3,AADR,"[GDF, FRI, MLPO, GAA, ICI, CLY, EUM, MUNI, DRW...",10
4,AAL,"[KOS, VAL, LPI, WYNN, MTDR, OIS, PDS, TMUS, DR...",10


## 20. Graph Statistics

In [28]:
total_edges = (

    len(company_publisher_edges)

    +

    len(company_event_edges)

    +

    len(company_signal_edges)

    +

    len(company_cluster_edges)

    +

    len(company_fingerprint_edges)

    +

    len(company_similarity_edges)

    +

    len(publisher_event_edges)

    +

    len(event_event_edges)

)

graph_statistics = pd.DataFrame({

    "Metric":[

        "Total Nodes",

        "Companies",

        "Publishers",

        "Events",

        "Signals",

        "Clusters",

        "Fingerprints",

        "Total Edges"

    ],

    "Value":[

        len(nodes),

        len(company_nodes),

        len(publisher_nodes),

        len(event_nodes),

        len(signal_nodes),

        len(cluster_nodes),

        len(fingerprint_nodes),

        total_edges

    ]

})

display(graph_statistics)

,Metric,Value
0,Total Nodes,7720
1,Companies,6619
2,Publishers,1047
3,Events,30
4,Signals,4
5,Clusters,12
6,Fingerprints,8
7,Total Edges,427023


## 21. Merge All Edges

In [29]:
edges = pd.concat(

    [

        company_publisher_edges,

        company_event_edges,

        company_signal_edges,

        company_cluster_edges,

        company_fingerprint_edges,

        company_similarity_edges,

        publisher_event_edges,

        event_event_edges

    ],

    ignore_index=True

)

print("Knowledge Graph Edges")

print(edges.shape)

display(edges.head())

Knowledge Graph Edges
(427023, 5)


,source,target,relation,weight,confidence
0,COMP_A,PUB_Above_Average_Odds,PUBLISHED_BY,2.0,0.867500
1,COMP_A,PUB_Accesswire,PUBLISHED_BY,1.0,0.748400
2,COMP_A,PUB_Allie_Wickman,PUBLISHED_BY,27.0,0.830659
3,COMP_A,PUB_Andrew_Efimoff,PUBLISHED_BY,2.0,0.866500
4,COMP_A,PUB_Angry_Bear,PUBLISHED_BY,2.0,0.738000


## 22. Save Nodes

In [30]:
nodes_output = os.path.join(

    GRAPH_PATH,

    "nodes.parquet"

)

nodes.to_parquet(

    nodes_output,

    index=False

)

print(nodes_output)

/content/drive/MyDrive/FinSight_AI/data/processed/knowledge_graph/nodes.parquet


## 23. Save Edges

In [31]:
edges_output = os.path.join(

    GRAPH_PATH,

    "edges.parquet"

)

edges.to_parquet(

    edges_output,

    index=False

)

print(edges_output)

/content/drive/MyDrive/FinSight_AI/data/processed/knowledge_graph/edges.parquet


## 24. Save Company Neighborhood

In [32]:
neighbor_output = os.path.join(

    GRAPH_PATH,

    "company_neighbors.parquet"

)

company_neighbors.to_parquet(

    neighbor_output,

    index=False

)

print(neighbor_output)

/content/drive/MyDrive/FinSight_AI/data/processed/knowledge_graph/company_neighbors.parquet


## 25. Save Graph Statistics

In [33]:
stats_output = os.path.join(

    GRAPH_PATH,

    "graph_statistics.parquet"

)

graph_statistics.to_parquet(

    stats_output,

    index=False

)

print(stats_output)

/content/drive/MyDrive/FinSight_AI/data/processed/knowledge_graph/graph_statistics.parquet


## 26. Summary

In [34]:
summary = pd.DataFrame({

    "Metric":[

        "Nodes",

        "Edges",

        "Companies",

        "Publishers",

        "Events",

        "Similarity Edges",

        "Output Folder"

    ],

    "Value":[

        len(nodes),

        len(edges),

        len(company_nodes),

        len(publisher_nodes),

        len(event_nodes),

        len(company_similarity_edges),

        GRAPH_PATH

    ]

})

display(summary)

,Metric,Value
0,Nodes,7720
1,Edges,427023
2,Companies,6619
3,Publishers,1047
4,Events,30
5,Similarity Edges,66190
6,Output Folder,/content/drive/MyDrive/FinSight_AI/data/proces...
